In [4]:
import os
from dotenv import load_dotenv
from pprint import pprint
load_dotenv()

from google.cloud import storage, bigquery

In [5]:
os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

'/home/hyderreza/codehub/ade-pipeline/keys/gcs-credentials.json'

In [6]:
# Helper functions
def get_year(blob):
    return blob.name.split("/")[3]

def scan_years(blobs): 
    return list({get_year(blob) for blob in blobs})

def update_uris(bucket, schema, years):
    return list(f"gs://{bucket}/cleaned/pq/{schema}/{year}/*.parquet" for year in years)

In [7]:
BUCKET_NAME = "zoomcamp-454219-ade-pipeline"
DATASET_ID = "ade_external"
SCHEMA = ['patient', 'reaction', 'drug']

def update_external_table_uris():
    storage_client = storage.Client()
    bigquery_client = bigquery.Client()

    # Update uris of external table per schema
    for s in SCHEMA:
        blobs = storage_client.list_blobs(BUCKET_NAME, prefix=f"cleaned/pq/{s}")
        years = scan_years(blobs)

        # Setting Configuration
        external_config = bigquery.ExternalConfig("PARQUET")
        external_config.source_uris = update_uris(BUCKET_NAME, s, years)
        external_config.autodetect = True

        table_ref = f"{bigquery_client.project}.{DATASET_ID}.ext_{s}"
        try:
            # Check if table exists
            table = bigquery_client.get_table(table_ref)
            table.external_data_configuration = external_config

            print(f"Table found: {table_ref}")
            print(f"Updating source_uris..")
            bigquery_client.update_table(table, ['external_data_configuration'])
            pprint(f"{table.external_data_configuration.source_uris}")
        except Exception as e:
            print("Table does not exist")
            print("Creating new table...")
            
            new_table = bigquery.Table(table_ref)
            new_table.external_data_configuration = external_config
            table = bigquery_client.create_table(new_table)

            print(f"Created table {table.project}.{table.dataset_id}.{table.table_id}")

update_external_table_uris()

Table found: zoomcamp-454219.ade_external.ext_patient
Updating source_uris..
("['gs://zoomcamp-454219-ade-pipeline/cleaned/pq/patient/2004/*.parquet', "
 "'gs://zoomcamp-454219-ade-pipeline/cleaned/pq/patient/2025/*.parquet', "
 "'gs://zoomcamp-454219-ade-pipeline/cleaned/pq/patient/2005/*.parquet']")
Table found: zoomcamp-454219.ade_external.ext_reaction
Updating source_uris..
("['gs://zoomcamp-454219-ade-pipeline/cleaned/pq/reaction/2004/*.parquet', "
 "'gs://zoomcamp-454219-ade-pipeline/cleaned/pq/reaction/2025/*.parquet', "
 "'gs://zoomcamp-454219-ade-pipeline/cleaned/pq/reaction/2005/*.parquet']")
Table found: zoomcamp-454219.ade_external.ext_drug
Updating source_uris..
("['gs://zoomcamp-454219-ade-pipeline/cleaned/pq/drug/2004/*.parquet', "
 "'gs://zoomcamp-454219-ade-pipeline/cleaned/pq/drug/2025/*.parquet', "
 "'gs://zoomcamp-454219-ade-pipeline/cleaned/pq/drug/2005/*.parquet']")
